<a href="https://colab.research.google.com/github/lautrevor/data-science-portfolio/blob/main/project-3-bc-collision-analysis/notebooks/01_load_and_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BC/National Collision Analysis — Data Load & Cleaning

**Purpose:** Load the 2019 and 2020 National Collision Database (NCDB) files, fix a formatting inconsistency between the two years, and combine them into a single table for SQL analysis.

**Source:** Transport Canada, National Collision Database — https://open.canada.ca/data/en/dataset/1eb9eba7-71d1-4b30-9fb1-30cbdab7e63a

**Scope:** 2019 (pre-pandemic baseline) + 2020 (COVID year), enabling a year-over-year comparison.

## Loading the data directly from GitHub
The raw CSVs are committed to this repo, so we pull them straight from GitHub — reproducible for anyone who runs this notebook, no manual upload needed.

In [3]:
library(readr)
library(dplyr)

base_url <- "https://raw.githubusercontent.com/lautrevor/data-science-portfolio/main/project-3-bc-collision-analysis/data/"

df_2019 <- read_csv(paste0(base_url, "NCD_2019.csv"))
df_2020 <- read_csv(paste0(base_url, "NCD_2020.csv"))

dim(df_2019)
dim(df_2020)

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Rows: 272301 Columns: 23
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (18): C_MNTH, C_WDAY, C_HOUR, C_CONF, C_RCFG, C_WTHR, C_RSUR, C_RALN, C_...
dbl  (5): C_YEAR, C_SEV, C_VEHS, V_ID, C_CASE

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 198745 Columns: 23
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (21): C_MNTH, C_WDAY, C_HOUR, C_VEHS, C_CONF, C_RCFG, C_WTHR, C_RSUR, C_...
dbl  (2): C_YEAR, C_SEV

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1] 272301     23

[1] 198745     23

## Fixing a data quality issue

The 2019 and 2020 files format the same codes differently — 2019 has no leading zeros (e.g. `C_MNTH` = `1`), 2020 is zero-padded (e.g. `C_MNTH` = `01`). Most columns also mix numeric codes with special text codes (`UU`/`NN`/`QQ`/`XX` for unknown/not-applicable), which forces R to read them as text.

Fix: convert every column except `P_SEX` (the only genuinely text column: F/M/N/U/X) to numeric. This makes `"01"` and `"1"` both become `1` (fixing the padding mismatch), and turns special/unknown codes into `NA`.

In [4]:
clean_numeric_columns <- function(df) {
  df %>%
    mutate(across(-P_SEX, ~ suppressWarnings(as.numeric(.))))
}

df_2019_clean <- clean_numeric_columns(df_2019)
df_2020_clean <- clean_numeric_columns(df_2020)

sort(unique(df_2019_clean$C_MNTH))
sort(unique(df_2020_clean$C_MNTH))

[1]  1  2  3  4  5  6  7  8  9 10 11 12

[1]  1  2  3  4  5  6  7  8  9 10 11 12

## Combining both years into one table
`C_YEAR` is already a real column in the data, so no manual year column is needed — just stack the rows.

In [5]:
combined <- bind_rows(df_2019_clean, df_2020_clean)

dim(combined)
table(combined$C_YEAR)

[1] 471046     23


  2019   2020 
272301 198745 

## Saving to SQLite
Writing the combined table into a `collisions.db` file for the repo's `/sql` folder.

In [6]:
install.packages("RSQLite")
library(DBI)
library(RSQLite)

con <- dbConnect(RSQLite::SQLite(), "collisions.db")
dbWriteTable(con, "collisions", combined, overwrite = TRUE)
dbDisconnect(con)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

